In [1]:
import os, sys, re, json, datetime, subprocess
from typing import Any, Dict, Optional, Literal, TypedDict, Annotated
import duckdb
from pydantic import BaseModel, Field, ConfigDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
class RouteParameters(BaseModel):
    # Ensures additionalProperties: false
    model_config = ConfigDict(extra="forbid")
    # Required but nullable (model must always emit the key; can be null)
    doctor_id: int | None
    date: str | None
    patient_id: int | None

In [4]:
from langchain_openai import ChatOpenAI

# -------------------------
# JSON-safe serialization
# -------------------------
def _json_safe(v: Any) -> Any:
    """Convert non-JSON-native values into JSON-serializable values."""
    if isinstance(v, datetime.datetime):
        return v.isoformat()
    if isinstance(v, datetime.date):
        return v.isoformat()
    if isinstance(v, datetime.time):
        return v.isoformat()
    if isinstance(v, dict):
        return {k: _json_safe(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_json_safe(x) for x in v]
    return v

# -------------------------
# DuckDB + helper functions
# -------------------------
def _fetchall_dicts(con: duckdb.DuckDBPyConnection, sql: str, params: Optional[tuple] = None):
    cur = con.execute(sql, params or ())
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return [dict(zip(cols, r)) for r in rows]

def _fetchone_dict(con: duckdb.DuckDBPyConnection, sql: str, params: Optional[tuple] = None):
    cur = con.execute(sql, params or ())
    row = cur.fetchone()
    if row is None:
        return None
    cols = [d[0] for d in cur.description]
    return dict(zip(cols, row))

# Demo DB (in-memory)
con = duckdb.connect(database=":memory:")

con.execute("""
CREATE TABLE patient (
  patient_id INTEGER,
  first_name TEXT,
  last_name TEXT,
  date_of_birth DATE,
  sex TEXT,
  email TEXT,
  phone TEXT,
  updated_at TIMESTAMP
);
""")

con.execute("""
CREATE TABLE medical_history (
  history_id INTEGER,
  patient_id INTEGER,
  condition TEXT,
  noted_at TIMESTAMP,
  notes TEXT
);
""")

con.execute("""
CREATE TABLE appointment (
  appointment_id INTEGER,
  doctor_id INTEGER,
  patient_id INTEGER,
  start_at TIMESTAMP,
  end_at TIMESTAMP,
  reason TEXT
);
""")

# Seed sample data
con.execute("""
INSERT INTO patient VALUES
  (1, 'Danielle', 'Garcia', DATE '1929-01-05', 'male', 'valerieboyd@example.com', '477-731-8063', TIMESTAMP '2026-01-17 23:38:47'),
  (2, 'Sam', 'Lee', DATE '1980-02-10', 'female', 'sam.lee@example.com', '555-222-1111', TIMESTAMP '2026-01-10 09:00:00');
""")

con.execute("""
INSERT INTO medical_history VALUES
  (101, 1, 'asthma', TIMESTAMP '2021-01-14 00:00:00', 'Uses inhaler PRN'),
  (102, 1, 'hypertension', TIMESTAMP '2019-06-01 12:00:00', 'Lifestyle counseling'),
  (201, 2, 'diabetes', TIMESTAMP '2020-03-20 08:30:00', 'Metformin started');
""")

con.execute("""
INSERT INTO appointment VALUES
  (1001, 10, 1, TIMESTAMP '2026-02-01 09:00:00', TIMESTAMP '2026-02-01 09:30:00', 'Follow-up'),
  (1002, 10, 2, TIMESTAMP '2026-02-01 10:00:00', TIMESTAMP '2026-02-01 10:20:00', 'Lab review'),
  (1003, 11, 1, TIMESTAMP '2026-02-01 13:00:00', TIMESTAMP '2026-02-01 13:45:00', 'Consult');
""")

# -------------------------
# "Tools": pre-written SQL
# -------------------------
def list_calendar(doctor_id: int, date: str) -> list[dict[str, Any]]:
    """Calendar view: list appointments for a doctor on a date (YYYY-MM-DD)."""
    rows = _fetchall_dicts(
        con,
        """
        SELECT a.appointment_id, a.doctor_id, a.patient_id,
               a.start_at, a.end_at, a.reason,
               p.first_name, p.last_name
        FROM appointment a
        JOIN patient p ON p.patient_id = a.patient_id
        WHERE a.doctor_id = ?
          AND CAST(a.start_at AS DATE) = CAST(? AS DATE)
        ORDER BY a.start_at ASC
        """,
        (doctor_id, date),
    )
    return _json_safe(rows)

def list_medical_history(patient_id: int) -> list[dict[str, Any]]:
    """Medical history view: list entries for a patient."""
    rows = _fetchall_dicts(
        con,
        """
        SELECT history_id, patient_id, condition, noted_at, notes
        FROM medical_history
        WHERE patient_id = ?
        ORDER BY noted_at DESC
        """,
        (patient_id,),
    )
    return _json_safe(rows)

def get_patient_details(patient_id: int) -> Optional[dict[str, Any]]:
    """Patient details view: fetch patient profile row."""
    row = _fetchone_dict(
        con,
        """
        SELECT patient_id, first_name, last_name, date_of_birth, sex, email, phone, updated_at
        FROM patient
        WHERE patient_id = ?
        """,
        (patient_id,),
    )
    return _json_safe(row) if row else None

# -------------------------
# Routing schema (STRICT)
# -------------------------
View = Literal["calendar", "medical_history", "patient_details", "unknown"]
Action = Literal["list", "ask_clarify"]

class Route(BaseModel):
    view: View = "unknown"
    action: Action = "ask_clarify"
    #parameters: Dict[str, Any] = Field(default_factory=dict)
    parameters: RouteParameters
    message: str = ""

# -------------------------
# Router: LLM or fallback
# -------------------------
ROUTER_SYSTEM = SystemMessage(content=(
    "You are a UI router. Output ONLY a JSON object that matches the Route schema.\n"
    "Allowed views: calendar, medical_history, patient_details, unknown.\n"
    "Allowed actions: list, ask_clarify.\n\n"
    "Rules:\n"
    "- If request is for calendar/schedule/appointments: view='calendar', action='list', parameters must include doctor_id (int) and date ('YYYY-MM-DD').\n"
    "- If request is for medical history: view='medical_history', action='list', parameters must include patient_id (int).\n"
    "- If request is for patient details/profile/demographics: view='patient_details', action='list', parameters must include patient_id (int).\n"
    "- If required parameters are missing, set action='ask_clarify' and ask a single direct question in message.\n"
    "- Do not include any extra keys or text outside the JSON."
))

def _extract_int(text: str, label: str) -> Optional[int]:
    # Supports "patient 12", "patient_id=12", "doctor: 10", etc.
    m = re.search(rf"\b{label}\b\s*[:=]?\s*(\d+)\b", text, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def _extract_date_yyyy_mm_dd(text: str) -> Optional[str]:
    m = re.search(r"\b(20\d{2}-\d{2}-\d{2})\b", text)
    return m.group(1) if m else None

#def _rule_based_router(user_text: str) -> Route:
#    t = user_text.strip()
#    tl = t.lower()
#
#    is_calendar = any(k in tl for k in ["calendar", "schedule", "appointments", "appt", "slot"])
#    is_history = any(k in tl for k in ["medical history", "history", "conditions", "diagnoses", "problem list"])
#    is_details = any(k in tl for k in ["patient details", "details", "profile", "demographics", "patient info"])
#
#    # Prefer explicit keywords; otherwise infer from ids + terms
#    if is_calendar:
#        doc_id = _extract_int(t, "doctor") or _extract_int(t, "doctor_id")
#        date = _extract_date_yyyy_mm_dd(t)
#        if doc_id is None and date is None:
#            return Route(view="calendar", action="ask_clarify", message="Which doctor_id and what date (YYYY-MM-DD)?")
#        if doc_id is None:
#            return Route(view="calendar", action="ask_clarify", parameters={"date": date}, message="What doctor_id?")
#        if date is None:
#            return Route(view="calendar", action="ask_clarify", parameters={"doctor_id": doc_id}, message="What date (YYYY-MM-DD)?")
#        return Route(view="calendar", action="list", parameters={"doctor_id": doc_id, "date": date}, message="")
#
#    if is_history:
#        pid = _extract_int(t, "patient") or _extract_int(t, "patient_id")
#        if pid is None:
#            return Route(view="medical_history", action="ask_clarify", message="What patient_id?")
#        return Route(view="medical_history", action="list", parameters={"patient_id": pid}, message="")
#
#    if is_details:
#        pid = _extract_int(t, "patient") or _extract_int(t, "patient_id")
#        if pid is None:
#            return Route(view="patient_details", action="ask_clarify", message="What patient_id?")
#        return Route(view="patient_details", action="list", parameters={"patient_id": pid}, message="")
#
#    # Heuristic: if they mention doctor_id + date, treat as calendar; if patient_id, ask which view
#    doc_id = _extract_int(t, "doctor") or _extract_int(t, "doctor_id")
#    date = _extract_date_yyyy_mm_dd(t)
#    pid = _extract_int(t, "patient") or _extract_int(t, "patient_id")
#
#    if doc_id is not None or date is not None:
#        if doc_id is None:
#            return Route(view="calendar", action="ask_clarify", parameters={"date": date}, message="What doctor_id?")
#        if date is None:
#            return Route(view="calendar", action="ask_clarify", parameters={"doctor_id": doc_id}, message="What date (YYYY-MM-DD)?")
#        return Route(view="calendar", action="list", parameters={"doctor_id": doc_id, "date": date})
#
#    if pid is not None:
#        return Route(
#            view="unknown",
#            action="ask_clarify",
#            parameters={"patient_id": pid},
#            message="Do you want medical_history or patient_details for this patient_id?"
#        )
#
#    return Route(view="unknown", action="ask_clarify", message="Do you want calendar, medical_history, or patient_details? Include the required id(s).")

router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Route)

# -------------------------
# LangGraph: state + nodes
# -------------------------
class AppState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    route: Dict[str, Any]
    data: Any

def router_node(state: AppState) -> AppState:
    user_text = state["messages"][-1].content if state.get("messages") else ""
    if router_llm is None:
        route_obj = _rule_based_router(user_text)
    else:
        route_obj = router_llm.invoke([ROUTER_SYSTEM, HumanMessage(content=user_text)])
    return {"route": route_obj.model_dump()}

def tool_exec_node(state: AppState) -> AppState:
    r = state.get("route") or {}
    view = r.get("view")
    action = r.get("action")
    params = r.get("parameters") or {}

    if action != "list":
        return {"data": None}

    # Map view+action to the correct pre-written SQL tool
    if view == "calendar":
        doctor_id = params.get("doctor_id")
        date = params.get("date")
        if doctor_id is None or date is None:
            return {"data": None}
        return {"data": list_calendar(int(doctor_id), str(date))}

    if view == "medical_history":
        patient_id = params.get("patient_id")
        if patient_id is None:
            return {"data": None}
        return {"data": list_medical_history(int(patient_id))}

    if view == "patient_details":
        patient_id = params.get("patient_id")
        if patient_id is None:
            return {"data": None}
        return {"data": get_patient_details(int(patient_id))}

    return {"data": None}

def emit_json_node(state: AppState) -> AppState:
    """
    Emit ONE JSON object as the assistant message.
    Your app can parse it and route to the correct view/action.
    """
    r = dict(state.get("route") or {})
    data = state.get("data", None)

    # Attach tool payload only when action=list and we have executed tools
    if r.get("action") == "list":
        r["parameters"] = r.get("parameters") or {}
        r["parameters"]["payload"] = data

        # If list action requested but payload is None (e.g., missing record),
        # convert to clarify to avoid silent failures.
        if data is None:
            r["action"] = "ask_clarify"
            r["message"] = r.get("message") or "I couldn't find data for that request. Check the id(s) and date."

    r = _json_safe(r)
    return {"messages": [AIMessage(content=json.dumps(r, ensure_ascii=False))]}

def _next_from_router(state: AppState) -> str:
    r = state.get("route") or {}
    return "tool_exec" if r.get("action") == "list" else "emit_json"

g = StateGraph(AppState)
g.add_node("router", router_node)
g.add_node("tool_exec", tool_exec_node)
g.add_node("emit_json", emit_json_node)

g.add_edge(START, "router")
g.add_conditional_edges("router", _next_from_router, ["tool_exec", "emit_json"])
g.add_edge("tool_exec", "emit_json")
g.add_edge("emit_json", END)

app = g.compile()

# -------------------------
# Helper to run a request
# -------------------------
def run(text: str) -> Dict[str, Any]:
    out = app.invoke({"messages": [HumanMessage(content=text)]})
    protocol_json = out["messages"][-1].content
    print(protocol_json)
    return json.loads(protocol_json)

print("LLM routing:", "ON" if router_llm is not None else "OFF (rule-based fallback)")



LLM routing: ON


In [5]:

print("\n--- Example: medical_history ---")
run("List medical history for patient_id 1")


--- Example: medical_history ---
{"view": "medical_history", "action": "list", "parameters": {"doctor_id": null, "date": null, "patient_id": 1, "payload": [{"history_id": 101, "patient_id": 1, "condition": "asthma", "noted_at": "2021-01-14T00:00:00", "notes": "Uses inhaler PRN"}, {"history_id": 102, "patient_id": 1, "condition": "hypertension", "noted_at": "2019-06-01T12:00:00", "notes": "Lifestyle counseling"}]}, "message": ""}


{'view': 'medical_history',
 'action': 'list',
 'parameters': {'doctor_id': None,
  'date': None,
  'patient_id': 1,
  'payload': [{'history_id': 101,
    'patient_id': 1,
    'condition': 'asthma',
    'noted_at': '2021-01-14T00:00:00',
    'notes': 'Uses inhaler PRN'},
   {'history_id': 102,
    'patient_id': 1,
    'condition': 'hypertension',
    'noted_at': '2019-06-01T12:00:00',
    'notes': 'Lifestyle counseling'}]},
 'message': ''}

In [6]:
print("\n--- Example: patient_details ---")
run("Show patient details for patient 2")


--- Example: patient_details ---
{"view": "patient_details", "action": "list", "parameters": {"doctor_id": null, "date": null, "patient_id": 2, "payload": {"patient_id": 2, "first_name": "Sam", "last_name": "Lee", "date_of_birth": "1980-02-10", "sex": "female", "email": "sam.lee@example.com", "phone": "555-222-1111", "updated_at": "2026-01-10T09:00:00"}}, "message": ""}


{'view': 'patient_details',
 'action': 'list',
 'parameters': {'doctor_id': None,
  'date': None,
  'patient_id': 2,
  'payload': {'patient_id': 2,
   'first_name': 'Sam',
   'last_name': 'Lee',
   'date_of_birth': '1980-02-10',
   'sex': 'female',
   'email': 'sam.lee@example.com',
   'phone': '555-222-1111',
   'updated_at': '2026-01-10T09:00:00'}},
 'message': ''}

In [7]:
print("\n--- Example: missing params (clarify) ---")
run("Show me the calendar for doctor 10")


--- Example: missing params (clarify) ---
{"view": "calendar", "action": "ask_clarify", "parameters": {"doctor_id": 10, "date": null, "patient_id": null}, "message": "What date do you want to see the calendar for?"}


{'view': 'calendar',
 'action': 'ask_clarify',
 'parameters': {'doctor_id': 10, 'date': None, 'patient_id': None},
 'message': 'What date do you want to see the calendar for?'}

In [9]:
print("\n--- Example: calendar ---")
run("Show the calendar for doctor_id 10 on 2026-12-01")


--- Example: calendar ---


KeyboardInterrupt: 